# 04 · Post hoc checks on the metadata audit

**Project:** Acquisition shortcuts in public cancer imaging (authors: see README.md)
**Notebook written:** 29 Oct 2026 · **Needs:** `results/02_tasks_features/` and `results/03_audit/` · **Plan:** section 7c

**Everything here is POST HOC.** These checks were decided on 29 Oct 2026 *after* the notebook 03 results were seen
(`00_LOG.md` section 5). They test how robust those results are. They do not replace them: the notebook 03 numbers stay the
pre-specified results.

**What this notebook does, in plain words**
0. **Reproduction check:** first it recomputes notebook 03's model exactly and confirms it gets the same numbers.
1. **Full word review (A + B):** notebook 02 checked only the 50 most common description words per collection. Here *every*
   word that the model could use (used by at least 3 patients in a task) was reviewed, without looking at the labels. A few
   missed words are now blocked (e.g. "guz" = Polish for tumor, "lym" = lymphoma). The main audit is then **rerun** with
   the corrected word list.
2. **Which kind of word (C):** every word is sorted by meaning into *clinical choice* (which body part was scanned, contrast
   and timing, why the scan was done), *technique* (slice thickness, kernel, sequence, scanner settings) or *unclear*. Then
   two stricter models are tested: one without the clinical-choice words, and a strict technical model with only scanner,
   software, slice/kernel and technique words.
3. **A proper chance test inside hospitals (D):** the within-hospital AUC of notebook 03 had no chance reference. Here the
   labels are shuffled *only inside each hospital* 999 times. For the CPTAC collections, whose hospitals are not known, the
   description language (Polish or not) is used as the grouping.

**No images are used.** Outputs go to `results/04_posthoc/`.

**How to run:** in Colab, *Runtime → Run all*, on a CPU runtime (High-RAM if offered). No GPU is needed. Sign in with
**the Google account that holds the project Drive folder** when Drive asks. Expect roughly an hour; if Colab disconnects, run all again: finished tasks
are loaded, not recomputed.

In [1]:
# 1. Setup: Drive and folders
import os, re, sys, json, time, hashlib, datetime, platform, warnings
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT = Path('/content/drive/MyDrive/Acquisition-Shortcut-Audit')
else:  # local code tests only
    PROJECT = Path(os.environ['ASA_PROJECT_DIR'])
assert PROJECT.exists(), (f'Project folder not found: {PROJECT}. '
                          'Is Colab signed in to the Drive of the Google account that holds the project Drive folder?')

IN02 = PROJECT / 'results' / '02_tasks_features'
IN03 = PROJECT / 'results' / '03_audit'
OUT = PROJECT / 'results' / '04_posthoc'
assert (IN02 / 'run_manifest.json').exists(), 'Run notebook 02 first: results/02_tasks_features/ is missing.'
assert (IN03 / 'run_manifest.json').exists(), 'Run notebook 03 first: results/03_audit/ is missing.'
OUT.mkdir(parents=True, exist_ok=True)

RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
print('Project folder:', PROJECT)
print('Outputs to    :', OUT)
print('Run started   :', RUN_STARTED, '(UTC)')

Mounted at /content/drive
Project folder: /content/drive/MyDrive/Acquisition-Shortcut-Audit
Outputs to    : /content/drive/MyDrive/Acquisition-Shortcut-Audit/results/04_posthoc
Run started   : 2026-10-28T19:04:36+00:00 (UTC)


In [2]:
# 2. Packages
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import rankdata
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
import joblib
from joblib import Parallel, delayed

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)
N_JOBS = -1

def show(df, n=30):
    """Display a table in Colab, or print it in a plain Python run."""
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())

def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

print('numpy', np.__version__, '| pandas', pd.__version__, '| scikit-learn', sklearn.__version__, '| CPUs', os.cpu_count())

numpy 2.1.3 | pandas 2.2.3 | scikit-learn 1.6.1 | CPUs 2


## Settings
The model settings are notebook 03's frozen settings (checked against its run record below). The word lists after them
are the post hoc decisions of 29 Oct 2026 (plan 7c), made from the word list alone, without looking at the labels.

In [3]:
# 3. Settings: notebook 03's frozen settings, then the post hoc word lists
SEED = 2026
N_SPLITS, N_REPEATS = 5, 5
N_BOOT = 2000
N_PERM = 999
MIN_COUNT = 3
FLAG_LOWER_CI, FLAG_Q = 0.60, 0.05
LR_PARAMS = dict(C=1.0, class_weight='balanced', max_iter=5000)

SCANNER_COLS = ['vendor', 'model', 'software_version', 'image_count', 'slice_thickness_mm', 'kernel', 'body_part',
                'protocol_text']
CATEGORICAL = {'vendor', 'model', 'software_version', 'kernel', 'body_part'}
LOG_NUMERIC = {'image_count'}
TEXT = {'protocol_text', 'words_clinical', 'words_technique', 'words_unclear', 'words_no_clinical'}

# (A) Words missed by the top-50 review of notebook 02 (rule 7a.3). Found in the full review of every usable word.
POSTHOC_LABEL_WORDS = {'guz',    # Polish: tumor
                       'spn',    # solitary pulmonary nodule ("PET/CT LUNG SPN")
                       'lym',    # lymphoma ("C-A-P LYM.")
                       'plc'}    # primary liver cancer ("PLC_TRIPHASIC_LIVER")
POSTHOC_SITE_WORDS = {'spsk1'}   # a Polish university hospital
POSTHOC_PROCESSING_WORDS = {'cad'}  # computer-aided detection output ("CAD~ sub1 cor")
# Words about why or where the study came from: moved to the care-pathway group (rule 7a.4), like 'screen' and 'outside'
POSTHOC_CARE_WORDS = {'diagnostic', 'cd', 'forfile', 'import', 'navigation'}

# (C) Word meaning. 'Clinical choice' = what the clinician decided to scan and how: body coverage, contrast and timing,
# purpose of the exam. 'Technique' = how the scanner acquired or reconstructed it. Every other word counts as 'unclear'
# (numbers, single letters with several meanings, modality names, function words).
COVERAGE_WORDS = {
    'ab', 'abd', 'abdome', 'abdomen', 'abdo', 'ap', 'a', 'p', 'cap', 'ctcap3k', 'ctap', 'ch', 'chest', '01routinechest',
    'thorax', 'torso', 'pel', 'pelv', 'pelve', 'pelvis', 'body', 'wb', 'wbs', 'whole', 'wholebody', 'head', 'brain', 'neck',
    'skull', 'base', 'thigh', 'to', 'through', 'top', 'bottom', 'mid', 'upper', 'hip', 'breast', 'bil', 'bilat',
    'bilateral', 'unilateral', 'kidney', 'kidneys', 'kids', 'renal', 'liver', 'liv', 'lungs', 'pancreas', 'panc',
    'pancreatic', '08pancreas', 'bladder', 'pleura', 'adrenal', 'prostaat', 'prostate', 'spine',
    # Polish
    'trzustka', 'brzuch', 'brzuchmiednica', 'brzuszna', 'brzusznej', 'brzuszn', 'brzuszne', 'brz', 'j.brzuszna',
    'j.brzusznej', 'j.brzusz', 'jama', 'jamy', 'jb', 'j', 'miednica', 'miednicy', 'mied', 'mie', 'mniejszej', 'malej',
    'klp', 'klatki', 'piersiowej', 'watroba', 'nerki'}
CONTRAST_PHASE_WORDS = {
    'contrast', 'cont', 'con', 'contr', 'contra', 'co', 'c', 'i', 'w', 'wo', 'o', 'with', 'without', 'wwo', 'noncon',
    'nonenh', 'noncontrast', 'nonconchest', 'enh', 'enhanced', 'iv', 'dye', 'omnipaque', 'ultravist', 'ionic', 'gad',
    'ce', 'c0n', '100ml', '100mls', '5cc', 'arterial', 'art', 'venous', 'venogram', 'portal', 'p.venous', 'pv', 'phase',
    'phases', 'phasic', 'biphase', '2phase', 'triphasic', 'tri', 'multiphase', 'dualphase', 'delay', 'delays', 'delayed',
    'pre', 'post', 'dyn', 'dynamic', 'dynamics', 'dyndist', 'dynviews', 'cortical', 'parenchymal', 'nephro',
    'collecting', 'system', 'perfusion', 'perme', 'permeabilit', 'urogram', 'urography', 'angio', 'angiogram', 'cta',
    'pha', 'bolus', 'water', 'dry', 'smartprep', 'smartp', 'tracker', 'augment', 'augmented',
    # Polish
    'kontrastem', 'kontrastu', 'kontr', 'wzmoc', 'faza', 'fazy', 'faz', 'fazowy', '3fazy', 'fases', 'fasesvol',
    'tetnicza', 'zylna', 'opoznione', 'bez', 'przed', 'dynamiczna'}
PURPOSE_WORDS = {'pe', 'pet', 'petct', 'ptct', 'pt', 'thoraxpetct1', 'reference', 'clinical', 'routine', 'stones',
                 '74178', 'functional', 'motor', 'language', 'cholangiografia', 'mrcp'}
TECHNIQUE_WORDS = {
    '1.25mm', '2.5mm', '5mm', '1mm', '7mm', '3x2', '5x5', '5x2.5', '2x', '100ma', '0.8sec', '30dir', '3d', '2d', 'fl3d',
    '4bval', '3scan', 'b20f', 'b30f', 'b30s', 'b31f', 'b31s', 'b40f', 'b41s', 'b45', 'b45f', 'i30f', 'i31s', 'i40f',
    'fc02', 'standard', 'std', 'stnd', 'soft', 'tissue', 'st', 'bone', 'detail', 'edge', 'window', 'miekka', 'thin',
    'thins', 'thick', 'slices', 'recon', 'helical', 'spiral', 'axial', 'axials', 'ax', 'sag', 'cor', 'tra', 'mode',
    'collimator', 'dose', 'low', 'hd', 'hr', 'hi', 'res', 'large', 'sm', 'med', 'isotropic', 'volume', 'vol', 'dmpr',
    'mpr', 'asir', 'asir30', 'gsi', 'dual', 'ma', 'smartma', 'auto', 'mm', 'fs', 'sat', 'fat', 'spair', 'vibrant',
    'vibe', 'thrive', 'ethrive', 'sense', 'asset', 't1', 't2', 'flair', 'tse', 'ep2d', 'diff', 'diffusie', 'dwi', 'dti',
    'mddw', 'mprage', 'wip', 'dr', 'idose', 'feet', 'first', 'prone', 'supine', 'adult', 'lmtd', 'bh', 'flow'}
CLINICAL_CHOICE_WORDS = COVERAGE_WORDS | CONTRAST_PHASE_WORDS | PURPOSE_WORDS
assert not (CLINICAL_CHOICE_WORDS & TECHNIQUE_WORDS), CLINICAL_CHOICE_WORDS & TECHNIQUE_WORDS

# Feature sets for part C
NO_CLINICAL_COLS = [c if c != 'protocol_text' else 'words_no_clinical' for c in SCANNER_COLS]
STRICT_COLS = ['vendor', 'model', 'software_version', 'slice_thickness_mm', 'kernel', 'words_technique']

# (D) Description language for the CPTAC collections, whose hospitals are not known. Unambiguous Polish words only.
POLISH_WORDS = {'tk', 'trzustka', 'jamy', 'jama', 'brzuch', 'brzuchmiednica', 'brzuszna', 'brzusznej', 'brzuszn',
                'brzuszne', 'brz', 'j.brzuszna', 'j.brzusznej', 'j.brzusz', 'miednica', 'miednicy', 'mied',
                'kontrastem', 'kontrastu', 'faza', 'fazy', 'fazowy', '3fazy', 'tetnicza', 'zylna', 'opoznione', 'bez',
                'przed', 'klatki', 'piersiowej', 'klp', 'tomografia', 'watroba', 'nerki', 'malej', 'mniejszej',
                'miekka', 'przegladowe', 'cholangiografia', 'wielo', 'lub', 'wzmoc', 'nowy', 'pacjent', 'dynamiczna',
                'guz', 'spsk1'}
LANGUAGE_TASK_PREFIX = 'CPTAC-'

# Local code tests only (ignored in Colab): shuffled labels, fewer tasks, fewer resamples
TEST = {}
if not IN_COLAB:
    TEST = {'shuffle_labels': os.environ.get('ASA_TEST_SHUFFLE_LABELS') == '1',
            'tasks': [t for t in os.environ.get('ASA_TEST_TASKS', '').split(';') if t]}
    N_PERM = int(os.environ.get('ASA_TEST_N_PERM', N_PERM))
    N_BOOT = int(os.environ.get('ASA_TEST_N_BOOT', N_BOOT))
    print('LOCAL TEST SETTINGS:', TEST, '| N_PERM', N_PERM, '| N_BOOT', N_BOOT)

In [4]:
# 4. Load and verify the inputs from notebooks 02 and 03
man02 = json.loads((IN02 / 'run_manifest.json').read_text())
bad = [f['file'] for f in man02['outputs'] if md5(IN02 / f['file']) != f['md5']]
assert not bad, f'These notebook 02 files changed since that run: {bad}'
man03 = json.loads((IN03 / 'run_manifest.json').read_text())
bad = [f['file'] for f in man03['outputs'] if md5(IN03 / f['file']) != f['md5']]
assert not bad, f'These notebook 03 files changed since that run: {bad}'
assert man03['in_colab'] and not man03['local_test_settings'], 'results/03_audit/ is not the real Colab run'
assert man03['input_notebook02_run'] == man02['run_started_utc'], 'notebook 03 was run on other notebook 02 outputs'
s03 = man03['settings']
assert (s03['SEED'], s03['N_SPLITS'], s03['N_REPEATS'], s03['MIN_COUNT'], s03['LR_PARAMS']) == \
       (SEED, N_SPLITS, N_REPEATS, MIN_COUNT, LR_PARAMS), 'model settings differ from notebook 03'
print('Notebook 02 outputs verified:', len(man02['outputs']), 'files, run of', man02['run_started_utc'])
print('Notebook 03 outputs verified:', len(man03['outputs']), 'files, run of', man03['run_started_utc'])
v03 = man03['versions']
same_versions = (v03['numpy'], v03['pandas'], v03['scikit-learn']) == (np.__version__, pd.__version__, sklearn.__version__)
print('Package versions', 'match notebook 03' if same_versions else
      f"DIFFER from notebook 03 (then numpy {v03['numpy']}, pandas {v03['pandas']}, scikit-learn {v03['scikit-learn']})")

tasks = pd.read_csv(IN02 / 'tasks_final.csv')
labels = pd.read_csv(IN02 / 'labels_long.csv')
feat0 = pd.read_csv(IN02 / 'patient_features.csv', low_memory=False)
toks0 = pd.read_csv(IN02 / 'patient_tokens.csv.gz')
sites = pd.read_csv(IN02 / 'patient_sites.csv')
a03 = pd.read_csv(IN03 / 'audit_primary.csv')
sites03 = pd.read_csv(IN03 / 'audit_sites.csv')
oof03 = pd.read_csv(IN03 / 'oof_predictions.csv.gz')

KEY = ['collection', 'PatientID', 'modality']
family = tasks[tasks.eligible].task_id.tolist()
assert family == man03['family'], 'the task family differs from notebook 03'
if TEST.get('tasks'):
    family = [t for t in family if t in TEST['tasks']]
print(f'{len(family)} tasks')

Notebook 02 outputs verified: 11 files, run of 2026-10-28T15:29:58+00:00
Notebook 03 outputs verified: 57 files, run of 2026-10-28T16:27:39+00:00
Package versions match notebook 03
26 tasks


## Part A · The full word review (post hoc, labels not used)
Every word that the model could use, i.e. used by at least 3 patients in at least one task, with the review decision and
its meaning class. Only patient counts are used here, never the labels.

In [5]:
# 5. Part A: word review table and the corrected word list
task_patients = labels[labels.task_id.isin(family)][KEY + ['task_id']]
tt = toks0.merge(task_patients, on=KEY)
per_task_n = tt.groupby(['category', 'token', 'task_id']).PatientID.nunique().reset_index(name='n')
usable = per_task_n[per_task_n.n >= MIN_COUNT]
review = (usable.groupby(['category', 'token'])
          .agg(max_patients_in_a_task=('n', 'max'), n_tasks=('task_id', 'nunique')).reset_index()
          .merge(tt.groupby('token').collection.agg(lambda s: ', '.join(sorted(set(s)))).rename('collections'),
                 left_on='token', right_index=True))

def decision(tok, cat):
    if tok in POSTHOC_LABEL_WORDS:
        return 'blocked post hoc: label'
    if tok in POSTHOC_SITE_WORDS:
        return 'blocked post hoc: site'
    if tok in POSTHOC_PROCESSING_WORDS:
        return 'blocked post hoc: processing'
    if tok in POSTHOC_CARE_WORDS and cat == 'protocol':
        return 'moved post hoc to care pathway'
    return 'kept'

def word_class(tok):
    if tok in COVERAGE_WORDS:
        return 'clinical choice: coverage'
    if tok in CONTRAST_PHASE_WORDS:
        return 'clinical choice: contrast or phase'
    if tok in PURPOSE_WORDS:
        return 'clinical choice: purpose'
    if tok in TECHNIQUE_WORDS:
        return 'technique'
    return 'unclear'

review['decision'] = [decision(t, c) for t, c in zip(review.token, review.category)]
review['word_class'] = [word_class(t) if c == 'protocol' and d == 'kept' else '' for t, c, d in
                        zip(review.token, review.category, review.decision)]
review = review.sort_values(['category', 'token'])
review.to_csv(OUT / 'word_review_full.csv', index=False)
print(f'{len(review)} usable words reviewed:')
show(review.decision.value_counts().rename('words').to_frame())
show(review[review.category == 'protocol'].word_class.value_counts().rename('protocol words').to_frame())

# corrected tokens: drop the newly blocked words, move the care-pathway words
blocked_now = POSTHOC_LABEL_WORDS | POSTHOC_SITE_WORDS | POSTHOC_PROCESSING_WORDS
toks1 = toks0[~toks0.token.isin(blocked_now)].copy()
toks1.loc[toks1.token.isin(POSTHOC_CARE_WORDS), 'category'] = 'care pathway'
changed = toks0[toks0.token.isin(blocked_now | POSTHOC_CARE_WORDS) & (toks0.category == 'protocol')]
print(f'Task patients whose protocol words change: '
      f'{changed.merge(task_patients[KEY].drop_duplicates(), on=KEY).drop_duplicates(KEY).shape[0]}')

def text_columns(tk):
    """One space-joined word string per patient for each word set."""
    prot = tk[tk.category == 'protocol'].copy()
    prot['cls'] = prot.token.map(word_class)
    out = {}
    def join(d):
        return d.groupby(KEY).token.agg(lambda s: ' '.join(sorted(set(s))))
    out['protocol_text'] = join(prot)
    out['words_clinical'] = join(prot[prot.cls.str.startswith('clinical choice')])
    out['words_technique'] = join(prot[prot.cls == 'technique'])
    out['words_unclear'] = join(prot[prot.cls == 'unclear'])
    out['words_no_clinical'] = join(prot[~prot.cls.str.startswith('clinical choice')])
    return pd.DataFrame(out).reset_index()

def build_features(tk):
    f = feat0.merge(text_columns(tk), on=KEY, how='left')
    for c in TEXT:
        f[c] = f[c].fillna('')
    for c in CATEGORICAL:
        f[c] = f[c].map(lambda v: str(v) if pd.notna(v) else np.nan).astype(object)
    return f

feat_orig = build_features(toks0)   # exactly notebook 03's inputs
feat_corr = build_features(toks1)   # with the corrected word list
site_of = sites.drop_duplicates('PatientID').set_index('PatientID').institution
lang_tokens = toks0[toks0.token.isin(POLISH_WORDS)].drop_duplicates(KEY)[KEY].assign(polish=True)

def task_data(tid, feat, shuffle=None):
    lab = labels[labels.task_id == tid]
    df = lab.merge(feat, on=KEY, how='left', validate='one_to_one')
    assert df.image_count.notna().all(), f'features missing for some patients of {tid}'
    classes = sorted(df.label.unique())
    y = df.label.map({c: i for i, c in enumerate(classes)}).to_numpy()
    if (TEST.get('shuffle_labels') if shuffle is None else shuffle):
        y = np.random.default_rng(999).permutation(y)
    df['site'] = df.PatientID.map(site_of)
    df = df.merge(lang_tokens, on=KEY, how='left')
    df['language'] = np.where(df.polish.eq(True), 'Polish', 'other')
    return df.reset_index(drop=True), y, classes

381 usable words reviewed:


,words
decision,
kept,374
moved post hoc to care pathway,3
blocked post hoc: label,3
blocked post hoc: processing,1


,protocol words
word_class,
unclear,125
technique,85
clinical choice: contrast or phase,76
clinical choice: coverage,62
clinical choice: purpose,12
,7


Task patients whose protocol words change: 65


## Model building blocks (identical to notebook 03)

In [6]:
# 6. Encoding, models, AUC, cross-validation, permutation test, within-group AUC
class TokenBinarizer(BaseEstimator, TransformerMixin):
    """One 0/1 column per word used by at least `min_count` training patients."""
    def __init__(self, min_count=MIN_COUNT):
        self.min_count = min_count
    def _docs(self, X):
        X = X.iloc[:, 0] if isinstance(X, pd.DataFrame) else pd.Series(np.asarray(X).ravel())
        return X.fillna('').astype(str).tolist()
    def fit(self, X, y=None):
        counts = {}
        for d in self._docs(X):
            for t in set(d.split()):
                counts[t] = counts.get(t, 0) + 1
        self.vocab_ = {t: i for i, t in enumerate(sorted(t for t, c in counts.items() if c >= self.min_count))}
        return self
    def transform(self, X):
        rows, cols = [], []
        docs = self._docs(X)
        for i, d in enumerate(docs):
            for t in set(d.split()):
                j = self.vocab_.get(t)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        return sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(docs), max(len(self.vocab_), 1)))

def column_transformer(cols):
    parts = []
    for c in cols:
        if c in CATEGORICAL:
            parts.append((c, Pipeline([('fill', SimpleImputer(strategy='constant', fill_value='missing',
                                                              keep_empty_features=True)),
                                       ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                min_frequency=MIN_COUNT, sparse_output=True))]), [c]))
        elif c in TEXT:
            parts.append((c, TokenBinarizer(), [c]))
        else:
            steps = [('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one'))] if c in LOG_NUMERIC else []
            steps += [('fill', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)),
                      ('scale', StandardScaler())]
            parts.append((c, Pipeline(steps), [c]))
    return ColumnTransformer(parts, remainder='drop', sparse_threshold=0.3)

def make_model(cols):
    return Pipeline([('prep', column_transformer(cols)), ('model', LogisticRegression(**LR_PARAMS))])

def auc_binary(y, s):
    """AUC by ranks (ties count one half); y in {0,1}."""
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(s)
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def auc(y, P):
    """Binary AUC, or macro one-vs-rest AUC for more than two classes. P: n x k probabilities."""
    k = P.shape[1]
    if k == 2:
        return auc_binary(y, P[:, 1])
    return float(np.nanmean([auc_binary((y == j).astype(int), P[:, j]) for j in range(k)]))

def repeated_splits(y, seed=SEED):
    out = []
    for r in range(N_REPEATS):
        skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=seed + r)
        out.append(list(skf.split(np.zeros(len(y)), y)))
    return out

def _fit_fold(df, y, cols, train, test, k):
    m = make_model(cols).fit(df.iloc[train], y[train])
    proba = np.zeros((len(test), k))
    proba[:, m.classes_] = m.predict_proba(df.iloc[test])
    return test, proba

def cv_oof(df, y, cols, splits=None):
    """Out-of-fold probabilities, shape (repeats, n, k)."""
    k = int(y.max()) + 1
    splits = splits or repeated_splits(y)
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold)(df, y, cols, tr, te, k) for r, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    for (r, _, _), (te, proba) in zip(jobs, res):
        P[r, te] = proba
    return P

def mean_auc(y, P):
    return float(np.mean([auc(y, P[r]) for r in range(P.shape[0])]))

def boot_ci(y, P, rng=None):
    """Percentile 95% CI of the repeat-averaged AUC over patient bootstrap resamples."""
    rng = rng or np.random.default_rng(SEED)
    k = P.shape[2]
    vals = []
    while len(vals) < N_BOOT:
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < k:
            continue
        vals.append(np.mean([auc(y[idx], P[r][idx]) for r in range(P.shape[0])]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def estimate(df, y, cols, splits=None):
    P = cv_oof(df, y, cols, splits)
    lo, hi = boot_ci(y, P)
    return {'auc': mean_auc(y, P), 'ci_low': lo, 'ci_high': hi}, P

# Permutation test of notebook 03 (7b.6): one fixed label-independent 5-fold split, encodings fitted once per fold
def _encoded_folds(df, cols):
    out = []
    for tr, te in KFold(N_SPLITS, shuffle=True, random_state=SEED).split(np.zeros(len(df))):
        ct = column_transformer(cols).fit(df.iloc[tr])
        out.append((tr, te, ct.transform(df.iloc[tr]), ct.transform(df.iloc[te])))
    return out

def _single_cv_proba(folds, y):
    k = int(y.max()) + 1
    P = np.zeros((len(y), k))
    for tr, te, Xtr, Xte in folds:
        m = LogisticRegression(**LR_PARAMS).fit(Xtr, y[tr])
        P[np.ix_(te, m.classes_)] = m.predict_proba(Xte)
    return P

def _perm_chunk(folds, ys):
    return [auc(yp, _single_cv_proba(folds, yp)) for yp in ys]

def permutation_test(df, y, cols):
    folds = _encoded_folds(df, cols)
    observed = auc(y, _single_cv_proba(folds, y))
    rng = np.random.default_rng(SEED)
    perms = [rng.permutation(y) for _ in range(N_PERM)]
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_perm_chunk)(folds, c) for c in chunks))
    p = (1 + int((null >= observed).sum())) / (1 + N_PERM)
    return observed, p, null

# Within-group AUC (7b.10c): share of positive-negative pairs from the same group that the score orders correctly
def within_site_auc(y, s, site):
    num, den, n_sites = 0.0, 0, 0
    for st in np.unique(site):
        m = site == st
        yy, ss = y[m], s[m]
        n1 = int(yy.sum()); n0 = len(yy) - n1
        if n1 and n0:
            r = rankdata(ss)
            num += r[yy == 1].sum() - n1 * (n1 + 1) / 2
            den += n1 * n0
            n_sites += 1
    return (num / den if den else np.nan), den, n_sites

def within_site_estimate(y, P, site):
    vals = [within_site_auc(y, P[r][:, 1], site)[0] for r in range(P.shape[0])]
    _, pairs, n_sites = within_site_auc(y, P[0][:, 1], site)
    if pairs == 0:   # no group holds both classes
        return {'auc': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n_pairs': 0, 'n_sites_with_pairs': 0}
    rng = np.random.default_rng(SEED)
    boots = []
    groups = [np.where(site == st)[0] for st in np.unique(site)]
    while len(boots) < N_BOOT:
        idx = np.concatenate([rng.choice(g, len(g), replace=True) for g in groups])
        v = np.mean([within_site_auc(y[idx], P[r][idx, 1], site[idx])[0] for r in range(P.shape[0])])
        if not np.isnan(v):
            boots.append(v)
    return {'auc': float(np.mean(vals)), 'ci_low': float(np.percentile(boots, 2.5)),
            'ci_high': float(np.percentile(boots, 97.5)), 'n_pairs': int(pairs), 'n_sites_with_pairs': n_sites}

# New in notebook 04 (part D): labels shuffled only inside each group, so the group-label link is kept
def _strata_chunk(folds, ys, strata, keep):
    return [within_site_auc(yp[keep], _single_cv_proba(folds, yp)[keep, 1], strata[keep])[0] for yp in ys]

def within_group_permutation_test(df, y, cols, strata, keep):
    """Exact test of 'no signal beyond the group'. The model is trained on all task patients; the statistic uses the
    patients in `keep`. Patients without a group form one extra shuffling group."""
    folds = _encoded_folds(df, cols)
    observed = within_site_auc(y[keep], _single_cv_proba(folds, y)[keep, 1], strata[keep])[0]
    rng = np.random.default_rng(SEED)
    groups = [np.where(strata == s)[0] for s in np.unique(strata)]
    perms = []
    for _ in range(N_PERM):
        yp = y.copy()
        for g in groups:
            yp[g] = y[rng.permutation(g)]
        perms.append(yp)
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_strata_chunk)(folds, c, strata, keep) for c in chunks))
    p = (1 + int((null >= observed).sum())) / (1 + N_PERM)
    return observed, p, null

def bh(p):
    """Benjamini-Hochberg q-values; missing p-values stay missing and are not counted."""
    p = np.asarray(p, float)
    q = np.full_like(p, np.nan)
    ok = np.where(~np.isnan(p))[0]
    order = ok[np.argsort(p[ok])]
    prev = 1.0
    for rank, j in reversed(list(enumerate(order, 1))):
        prev = min(prev, p[j] * len(ok) / rank)
        q[j] = prev
    return q

## Part 0 · Reproduction check
Real labels, notebook 03's inputs: the fixed-split AUC and every out-of-fold probability must match notebook 03.
(This part always uses the real labels, also in local code tests, because it only recomputes known numbers.)

In [7]:
# 7. Part 0: reproduce notebook 03
repro_rows = []
for tid in family:
    df, y, classes = task_data(tid, feat_orig, shuffle=False)
    folds = _encoded_folds(df, SCANNER_COLS)
    single = auc(y, _single_cv_proba(folds, y))
    P = cv_oof(df, y, SCANNER_COLS)
    o = oof03[oof03.task_id == tid]
    diffs = []
    for r in range(N_REPEATS):
        orr = o[o.repeat == r].set_index('PatientID').reindex(df.PatientID)
        old = orr[[f'p_{j}' for j in range(len(classes))]].to_numpy()
        diffs.append(np.abs(old - P[r]).max())
    ref = a03.set_index('task_id').loc[tid]
    repro_rows.append({'task_id': tid, 'nb03_single_cv_auc': ref.perm_observed_auc_single_cv, 'nb04_single_cv_auc': single,
                       'nb03_auc': ref.auc, 'nb04_auc': mean_auc(y, P), 'oof_max_abs_diff': float(max(diffs))})
repro = pd.DataFrame(repro_rows)
repro['single_cv_abs_diff'] = (repro.nb03_single_cv_auc - repro.nb04_single_cv_auc).abs()
repro['reproduced'] = (repro.single_cv_abs_diff < 1e-6) & (repro.oof_max_abs_diff < 1e-6)
repro.to_csv(OUT / 'reproduction_check.csv', index=False)
print(f'Reproduced exactly: {int(repro.reproduced.sum())} of {len(repro)} tasks '
      f'(largest probability difference {repro.oof_max_abs_diff.max():.2e})')
if not repro.reproduced.all():
    print('WARNING: notebook 03 was not reproduced exactly (package versions may have changed). '
          'Review the run log before interpreting any result below.')

Reproduced exactly: 26 of 26 tasks (largest probability difference 1.11e-16)


## Parts B, C, D · One pass per task
Each finished task is saved at once in `per_task/`; if Colab disconnects, run all again.

In [8]:
# 8. Run parts B (rerun with corrected words), C (word meaning) and D (chance test inside groups)
PER_TASK = OUT / 'per_task'
PER_TASK.mkdir(exist_ok=True)
SETTINGS_KEY = hashlib.md5(json.dumps(
    {'SEED': SEED, 'N_SPLITS': N_SPLITS, 'N_REPEATS': N_REPEATS, 'N_BOOT': N_BOOT, 'N_PERM': N_PERM,
     'MIN_COUNT': MIN_COUNT, 'LR': LR_PARAMS, 'TEST': TEST, 'inputs': man03['run_started_utc'],
     'words': [sorted(s) for s in (POSTHOC_LABEL_WORDS, POSTHOC_SITE_WORDS, POSTHOC_PROCESSING_WORDS,
                                   POSTHOC_CARE_WORDS, COVERAGE_WORDS, CONTRAST_PHASE_WORDS, PURPOSE_WORDS,
                                   TECHNIQUE_WORDS, POLISH_WORDS)]},
    sort_keys=True, default=str).encode()).hexdigest()

def _jsonable(o):
    return o.item() if hasattr(o, 'item') else str(o)

site_tasks = set(sites03.task_id)
rows_b, rows_c, rows_d, rows_lang = [], [], [], []
t_all = time.time()
for i, tid in enumerate(family, 1):
    t0 = time.time()
    stem = re.sub(r'\W+', '_', tid).strip('_')
    saved = PER_TASK / f'{stem}.json'
    if saved.exists():
        rec = json.loads(saved.read_text())
        if rec.get('settings_key') == SETTINGS_KEY:
            rows_b.append(rec['b']); rows_c.append(rec['c']); rows_d += rec['d']; rows_lang += rec['lang']
            print(f'{i:>2}/{len(family)} {tid:<44} loaded from an earlier run')
            continue
    n_d, n_lang = len(rows_d), len(rows_lang)

    # Part B: the primary analysis again, with the corrected word list
    df, y, classes = task_data(tid, feat_corr)
    k = len(classes)
    splits = repeated_splits(y)
    est, _ = estimate(df, y, SCANNER_COLS, splits)
    obs, p, null = permutation_test(df, y, SCANNER_COLS)
    changed_here = changed.merge(df[KEY], on=KEY).drop_duplicates(KEY).shape[0]
    b = {'task_id': tid, 'task_type': tasks.set_index('task_id').task_type[tid], 'n_patients': len(y),
         'classes': ' | '.join(f'{c}: {int((y == j).sum())}' for j, c in enumerate(classes)),
         'patients_with_changed_words': changed_here, **est, 'perm_p': p, 'null_mean_auc': float(null.mean())}

    # Part C: word meaning (corrected words)
    c = {'task_id': tid}
    for name, col in (('clinical_words', 'words_clinical'), ('technique_words', 'words_technique'),
                      ('unclear_words', 'words_unclear')):
        e, _ = estimate(df, y, [col], splits)
        c.update({f'{name}_alone_auc': e['auc'], f'{name}_alone_ci_low': e['ci_low'],
                  f'{name}_alone_ci_high': e['ci_high'], f'{name}_filled_fraction': float((df[col] != '').mean())})
    for name, cols in (('no_clinical', NO_CLINICAL_COLS), ('strict', STRICT_COLS)):
        e, _ = estimate(df, y, cols, splits)
        _, pp, nl = permutation_test(df, y, cols)
        c.update({f'{name}_auc': e['auc'], f'{name}_ci_low': e['ci_low'], f'{name}_ci_high': e['ci_high'],
                  f'{name}_perm_p': pp, f'{name}_null_mean_auc': float(nl.mean())})

    # Part D: chance test inside groups, on notebook 03's inputs (the pre-specified model)
    if k == 2:
        d0, y0, _ = task_data(tid, feat_orig)
        groupings = []
        if tid in site_tasks:
            groupings.append(('hospital', d0.site.astype(object).where(d0.site.notna(), '').astype(str).to_numpy()))
        if tid.startswith(LANGUAGE_TASK_PREFIX):
            groupings.append(('description language', d0.language.to_numpy()))
            for (lg, lb), n in d0.assign(label=[classes[j] for j in y0]).groupby(['language', 'label']).size().items():
                rows_lang.append({'task_id': tid, 'language': lg, 'label': lb, 'n_patients': int(n)})
        if groupings:
            P0 = cv_oof(d0, y0, SCANNER_COLS)
            for gname, strata in groupings:
                keep = strata != ''
                w = within_site_estimate(y0[keep], P0[:, keep], strata[keep])
                if w['n_pairs'] == 0:   # no group holds both classes: nothing to test
                    obs_w, p_w, null_w = np.nan, np.nan, np.array([np.nan])
                else:
                    obs_w, p_w, null_w = within_group_permutation_test(d0, y0, SCANNER_COLS, strata, keep)
                nb03_w = (sites03.set_index('task_id').within_site_auc.get(tid, np.nan) if gname == 'hospital'
                          else np.nan)
                rows_d.append({'task_id': tid, 'grouping': gname, 'n_patients': int(keep.sum()),
                               'n_groups': len(np.unique(strata[keep])),
                               'n_groups_with_both_classes': w['n_sites_with_pairs'], 'n_pairs': w['n_pairs'],
                               'within_auc': w['auc'], 'within_ci_low': w['ci_low'], 'within_ci_high': w['ci_high'],
                               'overall_auc_same_patients': mean_auc(y0[keep], P0[:, keep]),
                               'within_auc_fixed_split': obs_w,
                               'null_mean_within_auc': float(np.nanmean(null_w)) if np.isfinite(null_w).any() else np.nan,
                               'null_95th_within_auc': (float(np.nanpercentile(null_w, 95)) if np.isfinite(null_w).any()
                                                        else np.nan),
                               'perm_p': p_w, 'nb03_within_auc': nb03_w})

    rows_b.append(b); rows_c.append(c)
    saved.write_text(json.dumps({'settings_key': SETTINGS_KEY, 'b': b, 'c': c, 'd': rows_d[n_d:],
                                 'lang': rows_lang[n_lang:]}, default=_jsonable))
    print(f'{i:>2}/{len(family)} {tid:<44} B: AUC {est["auc"]:.3f} p={p:.3f} | strict AUC {c["strict_auc"]:.3f} '
          f'| D: {len(rows_d) - n_d} test(s)  ({time.time() - t0:.0f} s)')
print(f'Finished in {(time.time() - t_all) / 60:.1f} min')

 1/26 lung histology (LUAD vs LUSC)                B: AUC 0.827 p=0.001 | strict AUC 0.775 | D: 1 test(s)  (177 s)
 2/26 renal subtype (KIRC vs KIRP)                 B: AUC 0.842 p=0.001 | strict AUC 0.792 | D: 1 test(s)  (241 s)
 3/26 CPTAC-CCRCC | grade                          B: AUC 0.431 p=0.805 | strict AUC 0.487 | D: 1 test(s)  (155 s)
 4/26 CPTAC-CCRCC | stage                          B: AUC 0.392 p=0.847 | strict AUC 0.502 | D: 1 test(s)  (155 s)
 5/26 CPTAC-PDA | grade                            B: AUC 0.369 p=0.979 | strict AUC 0.389 | D: 1 test(s)  (151 s)
 6/26 CPTAC-PDA | stage                            B: AUC 0.840 p=0.001 | strict AUC 0.733 | D: 1 test(s)  (149 s)
 7/26 CPTAC-UCEC | grade                           B: AUC 0.370 p=0.971 | strict AUC 0.249 | D: 1 test(s)  (165 s)
 8/26 Duke-Breast-Cancer-MRI | ER status           B: AUC 0.501 p=0.465 | strict AUC 0.501 | D: 0 test(s)  (252 s)
 9/26 Duke-Breast-Cancer-MRI | HER2 status         B: AUC 0.496 p=0.361 | strict

## Multiple testing, flags and saving
Each test family gets its own Benjamini-Hochberg correction. Flags use rule 7a.2 (lower CI ≥ 0.60 and q < 0.05).

In [9]:
# 9. Corrections, flags, comparison with notebook 03, and saving
B = pd.DataFrame(rows_b)
B['perm_q_bh'] = bh(B.perm_p)
B['flagged'] = (B.ci_low >= FLAG_LOWER_CI) & (B.perm_q_bh < FLAG_Q)
ref = a03.set_index('task_id')
B['nb03_auc'] = B.task_id.map(ref.auc)
B['nb03_flagged'] = B.task_id.map(ref.flagged)
B['auc_change'] = B.auc - B.nb03_auc
B = B.sort_values('auc', ascending=False)
B.to_csv(OUT / 'posthoc_blocklist_rerun.csv', index=False)

C = pd.DataFrame(rows_c)
for name in ('no_clinical', 'strict'):
    C[f'{name}_perm_q_bh'] = bh(C[f'{name}_perm_p'])
    C[f'{name}_flagged'] = (C[f'{name}_ci_low'] >= FLAG_LOWER_CI) & (C[f'{name}_perm_q_bh'] < FLAG_Q)
C = C.set_index('task_id').loc[B.task_id].reset_index()
C.to_csv(OUT / 'posthoc_word_classes.csv', index=False)

D = pd.DataFrame(rows_d)
if len(D):
    D['perm_q_bh'] = np.nan
    for g, idx in D.groupby('grouping').groups.items():
        D.loc[idx, 'perm_q_bh'] = bh(D.loc[idx, 'perm_p'])
D.to_csv(OUT / 'posthoc_within_group_test.csv', index=False)
pd.DataFrame(rows_lang, columns=['task_id', 'language', 'label', 'n_patients']).to_csv(
    OUT / 'posthoc_language_counts.csv', index=False)

show(B[['task_id', 'auc', 'ci_low', 'ci_high', 'perm_q_bh', 'flagged', 'nb03_auc', 'nb03_flagged']], 40)
show(C[['task_id', 'clinical_words_alone_auc', 'technique_words_alone_auc', 'no_clinical_auc', 'no_clinical_flagged',
        'strict_auc', 'strict_flagged']], 40)
if len(D):
    show(D[['task_id', 'grouping', 'n_pairs', 'within_auc', 'within_ci_low', 'within_ci_high', 'null_mean_within_auc',
            'perm_p', 'perm_q_bh', 'nb03_within_auc']], 40)

,task_id,auc,ci_low,ci_high,perm_q_bh,flagged,nb03_auc,nb03_flagged
1,renal subtype (KIRC vs KIRP),0.842152,0.752397,0.917052,0.006500,True,0.846500,True
5,CPTAC-PDA | stage,0.840122,0.734428,0.922889,0.006500,True,0.840122,True
0,lung histology (LUAD vs LUSC),0.827269,0.744907,0.900635,0.006500,True,0.827269,True
13,LIDC-IDRI | primary vs metastatic,0.790242,0.695271,0.874832,0.006500,True,0.790242,True
22,TCGA-LIHC | grade,0.687654,0.570531,0.796015,0.037143,False,0.687500,False
12,LIDC-IDRI | benign vs malignant,0.687411,0.594264,0.775781,0.026000,False,0.688298,False
18,TCGA-BLCA | stage,0.626903,0.522498,0.730142,0.161778,False,0.624612,False
21,TCGA-KIRC | stage,0.610840,0.540316,0.675588,0.037143,False,0.614588,False
16,NSCLC Radiogenomics | histology,0.605458,0.518244,0.689409,0.215800,False,0.604272,False
17,PROSTATEx | clinically significant,0.589104,0.521098,0.659616,0.149500,False,0.589104,False


,task_id,clinical_words_alone_auc,technique_words_alone_auc,no_clinical_auc,no_clinical_flagged,strict_auc,strict_flagged
0,renal subtype (KIRC vs KIRP),0.734267,0.612712,0.820892,True,0.791746,True
1,CPTAC-PDA | stage,0.811854,0.640729,0.803647,True,0.733333,True
2,lung histology (LUAD vs LUSC),0.786486,0.596773,0.830900,True,0.774788,True
3,LIDC-IDRI | primary vs metastatic,0.803055,0.553808,0.746238,True,0.722937,True
4,TCGA-LIHC | grade,0.654630,0.593056,0.705093,False,0.705710,False
5,LIDC-IDRI | benign vs malignant,0.664775,0.564362,0.698138,True,0.630171,False
6,TCGA-BLCA | stage,0.546415,0.417147,0.546859,False,0.516556,False
7,TCGA-KIRC | stage,0.607518,0.571766,0.609586,False,0.547092,False
8,NSCLC Radiogenomics | histology,0.624973,0.632840,0.594396,False,0.614020,False
9,PROSTATEx | clinically significant,0.521151,0.517207,0.589019,False,0.517207,False


,task_id,grouping,n_pairs,within_auc,within_ci_low,within_ci_high,null_mean_within_auc,perm_p,perm_q_bh,nb03_within_auc
0,lung histology (LUAD vs LUSC),hospital,478,0.702929,0.530373,0.822237,0.456723,0.039,0.189000,0.702929
1,renal subtype (KIRC vs KIRP),hospital,244,0.588525,0.422460,0.742888,0.442627,0.138,0.331200,0.588525
2,CPTAC-CCRCC | grade,description language,485,0.449278,0.305275,0.600775,0.470609,0.735,0.994000,NaN
3,CPTAC-CCRCC | stage,description language,500,0.371600,0.238153,0.510078,0.470468,0.940,0.994000,NaN
4,CPTAC-PDA | grade,description language,510,0.351373,0.232269,0.481525,0.470404,0.964,0.994000,NaN
5,CPTAC-PDA | stage,description language,372,0.733871,0.576580,0.862110,0.460086,0.029,0.145000,NaN
6,CPTAC-UCEC | grade,description language,490,0.335102,0.216282,0.459691,0.450240,0.994,0.994000,NaN
7,NSCLC Radiogenomics | EGFR mutation,hospital,2450,0.406286,0.307616,0.504436,0.480668,0.884,0.973000,0.406286
8,NSCLC Radiogenomics | KRAS mutation,hospital,2303,0.403387,0.315918,0.491205,0.479735,0.930,0.973000,0.403387
9,NSCLC Radiogenomics | histology,hospital,2433,0.413728,0.320655,0.504592,0.480091,0.804,0.973000,0.413728


In [10]:
# 10. Run record (with a checksum for every output file)
outputs = []
for p in sorted(q for q in OUT.rglob('*') if q.is_file()):
    if p.name == 'run_manifest.json':
        continue
    n_rows = len(pd.read_csv(p, low_memory=False)) if (p.suffix == '.csv' or p.name.endswith('.csv.gz')) else None
    outputs.append({'file': p.relative_to(OUT).as_posix(), 'bytes': p.stat().st_size, 'rows': n_rows, 'md5': md5(p)})
manifest = {
    'settings_key': SETTINGS_KEY,
    'notebook': '04_posthoc_checks.ipynb', 'post_hoc': True, 'run_started_utc': RUN_STARTED,
    'run_finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'in_colab': IN_COLAB, 'local_test_settings': TEST, 'input_notebook02_run': man02['run_started_utc'],
    'input_notebook03_run': man03['run_started_utc'], 'family': family,
    'reproduced_notebook03': bool(repro.reproduced.all()), 'same_versions_as_notebook03': same_versions,
    'settings': {'SEED': SEED, 'N_SPLITS': N_SPLITS, 'N_REPEATS': N_REPEATS, 'N_BOOT': N_BOOT, 'N_PERM': N_PERM,
                 'MIN_COUNT': MIN_COUNT, 'FLAG_LOWER_CI': FLAG_LOWER_CI, 'FLAG_Q': FLAG_Q, 'LR_PARAMS': LR_PARAMS,
                 'NO_CLINICAL_COLS': NO_CLINICAL_COLS, 'STRICT_COLS': STRICT_COLS},
    'word_lists': {'POSTHOC_LABEL_WORDS': sorted(POSTHOC_LABEL_WORDS), 'POSTHOC_SITE_WORDS': sorted(POSTHOC_SITE_WORDS),
                   'POSTHOC_PROCESSING_WORDS': sorted(POSTHOC_PROCESSING_WORDS),
                   'POSTHOC_CARE_WORDS': sorted(POSTHOC_CARE_WORDS), 'COVERAGE_WORDS': sorted(COVERAGE_WORDS),
                   'CONTRAST_PHASE_WORDS': sorted(CONTRAST_PHASE_WORDS), 'PURPOSE_WORDS': sorted(PURPOSE_WORDS),
                   'TECHNIQUE_WORDS': sorted(TECHNIQUE_WORDS), 'POLISH_WORDS': sorted(POLISH_WORDS)},
    'versions': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                 'scikit-learn': sklearn.__version__, 'joblib': joblib.__version__, 'cpus': os.cpu_count()},
    'outputs': outputs,
}
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
show(pd.DataFrame(outputs)[lambda d: ~d.file.str.startswith('per_task/')])

,file,bytes,rows,md5
26,posthoc_blocklist_rerun.csv,6146,26.0,98ca6af1a977b5b8ec9031016631810d
27,posthoc_language_counts.csv,860,20.0,7bd0ec30a8b0b501d5c14098b7756a07
28,posthoc_within_group_test.csv,3769,17.0,f1e5423e0ba269a801b49f2f3015cd79
29,posthoc_word_classes.csv,12147,26.0,a2416d92d3e5f3cbfce0a0f08ce82c59
30,reproduction_check.csv,3646,26.0,ba0e689b10313252c4b4635ad9836fc7
31,word_review_full.csv,33121,381.0,51c7d31945e6ff195ebbdf3d156c8ea4


In [11]:
# 11. Plain-English summary, read back from the saved files
b = pd.read_csv(OUT / 'posthoc_blocklist_rerun.csv')
c = pd.read_csv(OUT / 'posthoc_word_classes.csv')
r = pd.read_csv(OUT / 'reproduction_check.csv')
print(f"- Notebook 03 reproduced exactly for {int(r.reproduced.sum())} of {len(r)} tasks.")
print(f"- Rerun with the corrected word list: {int(b.flagged.sum())} flagged (notebook 03: {int(b.nb03_flagged.sum())}); "
      f"largest AUC change {b.auc_change.abs().max():.3f}.")
print(f"- Without clinical-choice words: {int(c.no_clinical_flagged.sum())} flagged. "
      f"Strict technical model: {int(c.strict_flagged.sum())} flagged.")
print('- All of this is post hoc (plan 7c).')
print('\nNext: review the saved results and run log.')

- Notebook 03 reproduced exactly for 26 of 26 tasks.
- Rerun with the corrected word list: 4 flagged (notebook 03: 4); largest AUC change 0.004.
- Without clinical-choice words: 5 flagged. Strict technical model: 4 flagged.
- All of this is post hoc (plan 7c).

Next: review the saved results and run log.
